In [7]:
import git
import csv
import numpy as np
from cottools.scc import collect_scc, SccData
from cottools.filesolver import NameRepo
from concernbert.frontend import CdCalculator
from scipy.stats import pearsonr
from datetime import datetime
from collections import defaultdict
from concernbert.frontend import estimate_percentile

In [8]:
cd_calculator = CdCalculator("_models/EntityBERT-v3_train_nonldl-lr5e5-2_83-e3", "_cache")

In [9]:
REPO = "_repos/activemq"
repo = git.Repo(REPO, odbt=git.GitCmdObjectDB)
scc_data = collect_scc(REPO)
file_repo = NameRepo.parse_log(REPO, "main")


100%|██████████| 68806/68806 [00:15<00:00, 4408.38it/s]


Total edges:  10970
Used edges:   10912
Unused edges: 58
Total edges:  158
Used edges:   156
Unused edges: 2
Done


In [10]:
def load_bytes(repo: git.Repo, hexsha: str) -> bytes:
    return repo.odb.stream(bytes.fromhex(hexsha)).read()  # type: ignore

results = []

In [ ]:
latest_commit = file_repo.latest_commit()
for file_id in file_repo.file_ids(latest_commit):
    try:
        file_name = file_repo.file_name_by_id(file_repo.latest_commit(), file_id)
        cont_changes = file_repo.cont_changes_by_id(file_id)

        metrics = defaultdict(list)
        for rev in cont_changes:
            try:
                commit = repo.commit(rev)
                path = file_repo.file_name_by_id(rev, file_id)
                print("Path!!",path)
                obj = commit.tree.join(path)
                if not isinstance(obj, git.Blob):
                    continue

                content = load_bytes(repo, obj.hexsha)
                source = content.decode("utf-8", errors="ignore")
                cd = cd_calculator.calc_cd(source, pbar=False)
                print("Im merging code together")
                metrics['loc'].append(scc_data[obj.hexsha].loc)
                metrics['lloc'].append(scc_data[obj.hexsha].loc)  # reuse if lloc not separated
                metrics['entities'].append(cd.num_entities)
                metrics['intra_cd'].append(cd.intra_cd)
                metrics['inter_cd'].append(cd.inter_cd)
                metrics['estimate_percentile_inter_cd'].append(estimate_percentile(cd.inter_cd))
                metrics['estimate_percentile_intra_cd'].append(estimate_percentile(cd.intra_cd))
            except Exception:
                continue

        if len(metrics['loc']) < 2:
            continue  # not enough data points

        def safe_corr(x, y):
            try:
                return round(pearsonr(x, y)[0], 3)
            except:
                return ''
        results.append([
            file_name,
            len(metrics['loc']),
            safe_corr(metrics['loc'], metrics['intra_cd']),
            safe_corr(metrics['loc'], metrics['inter_cd']),
            safe_corr(metrics['lloc'], metrics['intra_cd']),
            safe_corr(metrics['lloc'], metrics['inter_cd']),
            safe_corr(metrics['entities'], metrics['intra_cd']),
            safe_corr(metrics['entities'], metrics['inter_cd']),
            safe_corr(metrics['entities'], metrics['inter_cd']),
            safe_corr(metrics['loc'], metrics['cdr_intra']),
            safe_corr(metrics['loc'], metrics['cdr_inter'])
        ])
         # Pretty print the current result
         #print("Most recent:",results[-1])
        print(f"Processing: {file_name}")
        print(f"# of Commits: {results[-1][1]}")
        print(f"corr(LOC, IntraCD): {results[-1][2]}")
        print(f"corr(LOC, InterCD): {results[-1][3]}")
        print(f"corr(LLOC, IntraCD): {results[-1][4]}")
        print(f"corr(LLOC, InterCD): {results[-1][5]}")
        print(f"corr(Entities, IntraCD): {results[-1][6]}")
        print(f"corr(Entities, InterCD): {results[-1][7]}")
        print(f"Mean CDR (Intra): {results[-1][8]}")
        print(f"Mean CDR (Inter): {results[-1][9]}")
        print("-" * 60)
    except Exception:
        continue
print("Im here!!!")
with open("correlation_metrics_over_time.csv", mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow([
        'File', '# of Commits',
        'corr(LOC, IntraCD)', 'corr(LOC, InterCD)',
        'corr(LLOC, IntraCD)', 'corr(LLOC, InterCD)',
        'corr(Entities, IntraCD)', 'corr(Entities, InterCD)',
         'Mean CDR (Intra)', 'Mean CDR (Inter)'
    ])
    writer.writerows(results)

print("CSV written to correlation_metrics_over_time.csv")


Path!! README.txt
Path!! README.txt
Path!! README.txt
Path!! README.txt
Path!! README.txt
Path!! README.txt
Path!! README.txt
Path!! README.txt
Path!! README.txt
Path!! README.txt
Path!! README.md
Path!! README.md
Path!! README.md
Path!! README.md
Path!! README.md
Path!! activemq-core/pom.xml


/Users/shawnblank/Documents/Drexel /Research/git/cottools/.venv/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/shawnblank/Documents/Drexel /Research/git/cottools/.venv/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activemq-core/pom.xml
Path!! activem

KeyboardInterrupt: 